<a href="https://colab.research.google.com/github/Mrzrm77/cnn-and-mobilenet-image-classification/blob/main/cnn_and_mobilenet_image_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("puneet6060/intel-image-classification")

print("Path to dataset files:", path)

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

In [ ]:
# Path to the training data directory
train_data_dir = path + "/seg_train/seg_train"

# Get the list of class labels (subdirectories)
class_labels = os.listdir(train_data_dir)

print("Distribution of labels in seg_train/seg_train:")
print("-------------------------------------------")

total_images = 0
for label in class_labels:
    label_path = os.path.join(train_data_dir, label)
    if os.path.isdir(label_path):
        num_images = len(os.listdir(label_path))
        print(f"{label}: {num_images} images")
        total_images += num_images

print("-------------------------------------------")
print(f"Total images in training dataset: {total_images}")

In [ ]:
# Path dataset
base_dir = train_data_dir
test_dir = path + "/seg_test/seg_test"

# Ukuran gambar dan batch size
IMG_SIZE = (150, 150)
BATCH_SIZE = 32

# Augmentasi data untuk training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    zoom_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    fill_mode='nearest'
)

# Data generator untuk validasi (tanpa augmentasi)
val_datagen = ImageDataGenerator(rescale=1./255,
                                 validation_split=0.2)

# Data generator untuk test
test_datagen = ImageDataGenerator(rescale=1./255)

# Membagi data train menjadi train & validation
train_generator = train_datagen.flow_from_directory(
    base_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    subset='training'
)

val_generator = val_datagen.flow_from_directory(
    base_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    subset='validation'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
import pandas as pd

# Get class labels
class_names = list(train_generator.class_indices.keys())

# Get counts for training set
train_counts = pd.Series(train_generator.classes).value_counts().sort_index()
train_labels = [class_names[i] for i in train_counts.index]

# Get counts for validation set
val_counts = pd.Series(val_generator.classes).value_counts().sort_index()
val_labels = [class_names[i] for i in val_counts.index]

# Get counts for test set
test_counts = pd.Series(test_generator.classes).value_counts().sort_index()
test_labels = [class_names[i] for i in test_counts.index]

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

sns.barplot(x=train_labels, y=train_counts.values, palette='viridis', ax=axes[0])
axes[0].set_title('Training Data Distribution')
axes[0].set_xlabel('Class Label')
axes[0].set_ylabel('Number of Images')
axes[0].tick_params(axis='x', rotation=45)

sns.barplot(x=val_labels, y=val_counts.values, palette='viridis', ax=axes[1])
axes[1].set_title('Validation Data Distribution')
axes[1].set_xlabel('Class Label')
axes[1].tick_params(axis='x', rotation=45)

sns.barplot(x=test_labels, y=test_counts.values, palette='viridis', ax=axes[2])
axes[2].set_title('Test Data Distribution')
axes[2].set_xlabel('Class Label')
axes[2].tick_params(axis='x', rotation=45)

plt.suptitle('Distribution of Images Across Datasets', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
labels = {value: key for key, value in train_generator.class_indices.items()}

print("Label Mappings for classes present in the training and validation datasets\n")
for key, value in labels.items():
    print(f"{key} : {value}")

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=5, figsize=(15, 12))
idx = 0

for i in range(2):
    for j in range(5):
        label = labels[np.argmax(train_generator[0][1][idx])]
        ax[i, j].set_title(f"{label}")
        ax[i, j].imshow(train_generator[0][0][idx][:, :, :])
        ax[i, j].axis("off")
        idx += 1

plt.tight_layout()
plt.suptitle("Sample Training Images", fontsize=21)
plt.show()

# Cnn Model

In [ ]:
cnn_model = Sequential([
    Conv2D(128, (5,5), activation='relu', input_shape=(150,150,3)),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(32, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(train_generator.num_classes, activation='softmax')
])

cnn_model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

cnn_model.summary()

In [ ]:
# callback
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, verbose=1)


In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
cnn_model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
history_cnn = cnn_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50,
    callbacks=[early_stop, reduce_lr]
)

In [ ]:
train_accuracy = history_cnn.history['accuracy']
val_accuracy = history_cnn.history['val_accuracy']

train_loss = history_cnn.history['loss']
val_loss = history_cnn.history['val_loss']


In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=1, figsize=(12, 10))

ax[0].set_title('Training Accuracy vs. Epochs')
ax[0].plot(train_accuracy, 'o-', label='Train Accuracy')
ax[0].plot(val_accuracy, 'o-', label='Validation Accuracy')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Accuracy')
ax[0].legend(loc='best')

ax[1].set_title('Training/Validation Loss vs. Epochs')
ax[1].plot(train_loss, 'o-', label='Train Loss')
ax[1].plot(val_loss, 'o-', label='Validation Loss')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Loss')
ax[1].legend(loc='best')

plt.tight_layout()
plt.show()

In [ ]:
predictions = cnn_model.predict(test_generator)

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=5, figsize=(12, 10))
idx = 0

for i in range(2):
    for j in range(5):
        predicted_label = labels[np.argmax(predictions[idx])]
        ax[i, j].set_title(f"{predicted_label}")
        ax[i, j].imshow(test_generator[0][0][idx])
        ax[i, j].axis("off")
        idx += 1

plt.tight_layout()
plt.suptitle("Test Dataset Predictions", fontsize=20)
plt.show()

In [ ]:
test_loss, test_accuracy = cnn_model.evaluate(test_generator, batch_size=BATCH_SIZE)


In [ ]:
print(f"Test Loss:     {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

In [ ]:
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

In [ ]:
cf_mtx = confusion_matrix(y_true, y_pred)

group_counts = ["{0:0.0f}".format(value) for value in cf_mtx.flatten()]
group_percentages = ["{0:.2%}".format(value) for value in cf_mtx.flatten()/np.sum(cf_mtx)]
box_labels = [f"{v1}\n({v2})" for v1, v2 in zip(group_counts, group_percentages)]
box_labels = np.asarray(box_labels).reshape(6, 6)

plt.figure(figsize = (12, 10))
sns.heatmap(cf_mtx, xticklabels=labels.values(), yticklabels=labels.values(),
           cmap="YlGnBu", fmt="", annot=box_labels)
plt.xlabel('Predicted Classes')
plt.ylabel('True Classes')
plt.show()

In [ ]:
print(classification_report(y_true, y_pred, target_names=labels.values()))

In [ ]:
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt
import os
import random # Import the random module

# Path to the directory containing images to predict
prediction_dir = path + "/seg_pred/seg_pred"

# Get a list of all files in the prediction directory
image_files = [f for f in os.listdir(prediction_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]

# Select 10 random images, or fewer if there aren't enough
num_images_to_predict = min(10, len(image_files))
images_to_predict = random.sample(image_files, num_images_to_predict)

if not images_to_predict:
    print(f"No image files found in {prediction_dir}.")
else:
    print(f"Predicting for {len(images_to_predict)} random images from {prediction_dir} using CNN and ResNet50:")

    plt.figure(figsize=(20, 15)) # Adjust figure size for two rows of predictions
    for i, img_filename in enumerate(images_to_predict):
        img_path = os.path.join(prediction_dir, img_filename)

        # Load the image
        img = image.load_img(img_path, target_size=IMG_SIZE)

        # Preprocess the image
        img_array = image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0) # Add batch dimension
        img_array /= 255.0 # Rescale to match training data preprocessing

        # Make prediction with CNN model
        cnn_prediction = cnn_model.predict(img_array, verbose=0)
        cnn_predicted_class_index = np.argmax(cnn_prediction)
        class_labels = list(train_generator.class_indices.keys())
        cnn_predicted_class_label = class_labels[cnn_predicted_class_index]

        # Display the image and CNN prediction
        plt.subplot(4, 5, i + 1) # 4 rows, 5 columns: first row for images
        plt.imshow(img)
        plt.title(f"CNN: {cnn_predicted_class_label}")
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# Mobilenet

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(150,150, 3),
    include_top=False,
    weights='imagenet'
)

In [ ]:
# Using the base model as a Feature Extractor
base_model.trainable = False

In [ ]:
mobilenet_model = tf.keras.Sequential([
    base_model,  # Feature Extractor
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(6, activation='softmax')
])
mobilenet_model.summary()

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
# Compile
mobilenet_model.compile(loss='categorical_crossentropy',
              optimizer=optimizer,
              metrics=['accuracy'])

# Fitting
history_mobilenet = mobilenet_model.fit(train_generator,
                    validation_data=val_generator,
                    epochs=50,
                    callbacks=[early_stop,reduce_lr])

In [ ]:
train_accuracy = history_mobilenet.history['accuracy']
val_accuracy = history_mobilenet.history['val_accuracy']

train_loss = history_mobilenet.history['loss']
val_loss = history_mobilenet.history['val_loss']

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=1, figsize=(12, 10))

ax[0].set_title('Training Accuracy vs. Epochs')
ax[0].plot(train_accuracy, 'o-', label='Train Accuracy')
ax[0].plot(val_accuracy, 'o-', label='Validation Accuracy')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Accuracy')
ax[0].legend(loc='best')

ax[1].set_title('Training/Validation Loss vs. Epochs')
ax[1].plot(train_loss, 'o-', label='Train Loss')
ax[1].plot(val_loss, 'o-', label='Validation Loss')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Loss')
ax[1].legend(loc='best')

plt.tight_layout()
plt.show()

In [ ]:
predictions2 = mobilenet_model.predict(test_generator)

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=5, figsize=(12, 10))
idx = 0

for i in range(2):
    for j in range(5):
        predicted_label = labels[np.argmax(predictions2[idx])]
        ax[i, j].set_title(f"{predicted_label}")
        ax[i, j].imshow(test_generator[0][0][idx])
        ax[i, j].axis("off")
        idx += 1

plt.tight_layout()
plt.suptitle("Test Dataset Predictions", fontsize=20)
plt.show()

In [ ]:
test_loss, test_accuracy = mobilenet_model.evaluate(test_generator, batch_size=BATCH_SIZE)


In [ ]:
print(f"Test Loss:     {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

In [ ]:
y_pred = np.argmax(predictions2, axis=1)
y_true = test_generator.classes

In [ ]:
cf_mtx = confusion_matrix(y_true, y_pred)

group_counts = ["{0:0.0f}".format(value) for value in cf_mtx.flatten()]
group_percentages = ["{0:.2%}".format(value) for value in cf_mtx.flatten()/np.sum(cf_mtx)]
box_labels = [f"{v1}\n({v2})" for v1, v2 in zip(group_counts, group_percentages)]
box_labels = np.asarray(box_labels).reshape(6, 6)

plt.figure(figsize = (12, 10))
sns.heatmap(cf_mtx, xticklabels=labels.values(), yticklabels=labels.values(),
           cmap="YlGnBu", fmt="", annot=box_labels)
plt.xlabel('Predicted Classes')
plt.ylabel('True Classes')
plt.show()

In [ ]:
print(classification_report(y_true, y_pred, target_names=labels.values()))


In [ ]:
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt
import os
import random # Import the random module

# Path to the directory containing images to predict
prediction_dir = path + "/seg_pred/seg_pred"

# Get a list of all files in the prediction directory
image_files = [f for f in os.listdir(prediction_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]

# Select 10 random images, or fewer if there aren't enough
num_images_to_predict = min(10, len(image_files))
images_to_predict = random.sample(image_files, num_images_to_predict)

if not images_to_predict:
    print(f"No image files found in {prediction_dir}.")
else:
    print(f"Predicting for {len(images_to_predict)} random images from {prediction_dir} using Mobilenet:")

    plt.figure(figsize=(20, 15)) # Adjust figure size for two rows of predictions
    for i, img_filename in enumerate(images_to_predict):
        img_path = os.path.join(prediction_dir, img_filename)

        # Load the image
        img = image.load_img(img_path, target_size=IMG_SIZE)

        # Preprocess the image
        img_array = image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0) # Add batch dimension
        img_array /= 255.0 # Rescale to match training data preprocessing

        # Make prediction with CNN model
        resnet_pred = mobilenet_model.predict(img_array, verbose=0)
        resnet_pred_index = np.argmax(resnet_pred)
        class_labels = list(train_generator.class_indices.keys())
        cnn_predicted_class_label = class_labels[resnet_pred_index]

        # Display the image and CNN prediction
        plt.subplot(4, 5, i + 1) # 4 rows, 5 columns: first row for images
        plt.imshow(img)
        plt.title(f"Mobilenet: {cnn_predicted_class_label}")
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# Download Model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Define the paths to save the models in Google Drive
cnn_model_save_path = '/content/drive/MyDrive/CVproject/cnn_model.keras'
mobilenet_save_path = '/content/drive/MyDrive/CVproject/mobilenet_model.keras'

# Save the CNN model
cnn_model.save(cnn_model_save_path)
print(f"CNN model saved to: {cnn_model_save_path}")

mobilenet_model.save(mobilenet_save_path)
print(f"Mobilenet model saved to: {mobilenet_save_path}")